## Installing required dependencies

In [ ]:
%pip install -U langgraph langchain langchain-community langchain-text-splitters langchain-google-genai bs4 pydantic tiktoken

## Setting the API KEYS using Gemini

In [ ]:
import os
from getpass import getpass

os.environ["GOOGLE_API_KEY"] = getpass("Enter API Key: ")

## Preprocessing the documents from the ones given in the tutorial page.

In [ ]:
from langchain_community.document_loaders import WebBaseLoader

urls = [
    "https://lilianweng.github.io/posts/2024-11-28-reward-hacking/",
    "https://lilianweng.github.io/posts/2024-07-07-hallucination/",
    "https://lilianweng.github.io/posts/2024-04-12-diffusion-video/",
]

docs = [WebBaseLoader(url).load() for url in urls]

# Preview first 500 chars of first doc
print(docs[0][0].page_content.strip()[:500])

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Flatten list of lists
docs_list = [item for sublist in docs for item in sublist]

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=100, chunk_overlap=50
)
doc_splits = text_splitter.split_documents(docs_list)

print(f"Total chunks: {len(doc_splits)}")
print("\nFirst chunk:")
print(doc_splits[0].page_content.strip())

## Creating the vector store and retriever tool using HuggingFaceEmbeddings

In [ ]:
%pip install -U langchain-huggingface sentence-transformers

from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

print("Embeddings loaded ✓")

In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_google_genai import GoogleGenerativeAIEmbeddings


vectorstore = InMemoryVectorStore.from_documents(
    documents=doc_splits,
    embedding=embeddings
)
retriever = vectorstore.as_retriever()

print("Vector store created ✓")

In [ ]:
from langchain.tools import tool

@tool
def retrieve_blog_posts(query: str) -> str:
    """Search and return information about Lilian Weng blog posts."""
    docs = retriever.invoke(query)
    return "\n\n".join([doc.page_content for doc in docs])

retriever_tool = retrieve_blog_posts

# Quick test
result = retriever_tool.invoke({"query": "types of reward hacking"})
print(result[:500])

## Building the LLM nodes with Gemini 2.5 Flash

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import MessagesState

# Main LLM used for generation and rewriting
response_model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

# Grader LLM (same model, used with structured output)
grader_model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

print("Models loaded ✓")

### Node 1 — Generate query or respond directly

In [ ]:
def generate_query_or_respond(state: MessagesState):
    """Call the model to generate a response or decide to retrieve."""
    response = (
        response_model
        .bind_tools([retriever_tool])
        .invoke(state["messages"])
    )
    return {"messages": [response]}

# Test: simple greeting (should NOT call the tool)
result = generate_query_or_respond({"messages": [{"role": "user", "content": "hello!"}]})
result["messages"][-1].pretty_print()

In [ ]:
# Test: question requiring retrieval (SHOULD call the retriever tool)
result = generate_query_or_respond({
    "messages": [{
        "role": "user",
        "content": "What does Lilian Weng say about types of reward hacking?"
    }]
})
result["messages"][-1].pretty_print()

### Node 2 — Grade documents for relevance

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

GRADE_PROMPT = (
    "You are a grader assessing relevance of a retrieved document to a user question. \n "
    "Here is the retrieved document: \n\n {context} \n\n"
    "Here is the user question: {question} \n"
    "If the document contains keyword(s) or semantic meaning related to the user question, grade it as relevant. \n"
    "Give a binary score 'yes' or 'no' score to indicate whether the document is relevant to the question."
)


class GradeDocuments(BaseModel):
    """Grade documents using a binary score for relevance check."""
    binary_score: str = Field(
        description="Relevance score: 'yes' if relevant, or 'no' if not relevant"
    )


def grade_documents(
    state: MessagesState,
) -> Literal["generate_answer", "rewrite_question"]:
    """Determine whether the retrieved documents are relevant to the question."""
    question = state["messages"][0].content
    context = state["messages"][-1].content

    prompt = GRADE_PROMPT.format(question=question, context=context)
    response = (
        grader_model
        .with_structured_output(GradeDocuments)
        .invoke([{"role": "user", "content": prompt}])
    )
    score = response.binary_score
    print(f"  [Grader] Score: {score}")

    if score == "yes":
        return "generate_answer"
    else:
        return "rewrite_question"


print("grade_documents defined ✓")

### Node 3 — Rewrite question

In [ ]:
from langchain_core.messages import HumanMessage

REWRITE_PROMPT = (
    "Look at the input and try to reason about the underlying semantic intent / meaning.\n"
    "Here is the initial question:"
    "\n ------- \n"
    "{question}"
    "\n ------- \n"
    "Formulate an improved question:"
)


def rewrite_question(state: MessagesState):
    """Rewrite the original user question to improve retrieval."""
    messages = state["messages"]
    question = messages[0].content
    prompt = REWRITE_PROMPT.format(question=question)
    response = response_model.invoke([{"role": "user", "content": prompt}])
    return {"messages": [HumanMessage(content=response.content)]}


### Node 4 — Generate final answer

In [ ]:
GENERATE_PROMPT = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer the question. "
    "If you don't know the answer, just say that you don't know. "
    "Use three sentences maximum and keep the answer concise.\n"
    "Question: {question} \n"
    "Context: {context}"
)


def generate_answer(state: MessagesState):
    """Generate a concise answer from retrieved context."""
    question = state["messages"][0].content
    context = state["messages"][-1].content
    prompt = GENERATE_PROMPT.format(question=question, context=context)
    response = response_model.invoke([{"role": "user", "content": prompt}])
    return {"messages": [response]}


## Visualizing our model with the LangGraph

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import InMemorySaver

workflow = StateGraph(MessagesState)

# -------------------------------
# Human-in-the-Loop Node (simple input() based)
# -------------------------------
def human_in_the_loop(state: MessagesState) -> Literal["generate_answer", "grade_documents_node"]:
    """Pause and ask the human if the retrieved docs look helpful."""
    question = state["messages"][0].content
    context = state["messages"][-1].content

    print("\n" + "=" * 60)
    print("HUMAN REVIEW REQUIRED")
    print("=" * 60)
    print(f"\nOriginal Question:\n{question}")
    print(f"\nRetrieved Documents:\n{context[:1000]}...")
    print("\n" + "=" * 60)
    print("Are the retrieved documents helpful enough to generate an answer?")
    print("  Type 'yes'    → generate answer directly")
    print("  Type 'unsure' → send to grading node first")
    print("=" * 60)

    while True:
        human_input = input("\nYour choice: ").strip().lower()
        if human_input == "yes":
            print("  [Human] Decision: generate directly ✓")
            return "generate_answer"
        elif human_input == "unsure":
            print("  [Human] Decision: send to grading ✓")
            return "grade_documents_node"
        else:
            print("  Invalid input. Please type exactly 'yes' or 'unsure'.")

# -------------------------------
# Add Nodes
# -------------------------------
workflow.add_node("generate_query_or_respond", generate_query_or_respond)
workflow.add_node("retrieve", ToolNode([retriever_tool]))
workflow.add_node("rewrite_question", rewrite_question)
workflow.add_node("generate_answer", generate_answer)
workflow.add_node("grade_documents_node", lambda state: state)

# -------------------------------
# Entry Point
# -------------------------------
workflow.add_edge(START, "generate_query_or_respond")

# -------------------------------
# After LLM: call tool or end directly
# -------------------------------
workflow.add_conditional_edges(
    "generate_query_or_respond",
    tools_condition,
    {"tools": "retrieve", END: END},
)

# -------------------------------
# After retrieve: human decides
# -------------------------------
workflow.add_conditional_edges(
    "retrieve",
    human_in_the_loop,
    {
        "generate_answer": "generate_answer",
        "grade_documents_node": "grade_documents_node",
    }
)

# -------------------------------
# After human says "unsure": grade
# -------------------------------
workflow.add_conditional_edges(
    "grade_documents_node",
    grade_documents,
    {
        "generate_answer": "generate_answer",
        "rewrite_question": "rewrite_question",
    }
)

# -------------------------------
# Final Edges
# -------------------------------
workflow.add_edge("generate_answer", END)
workflow.add_edge("rewrite_question", "generate_query_or_respond")

# -------------------------------
# Compile (NO checkpointer needed for input()-based pausing)
# -------------------------------
graph = workflow.compile()

print("Graph compiled with Human-in-the-Loop ✓")

In [ ]:
# Visualize the graph
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))

## Running Our RAG Model

In [ ]:
# Question requiring retrieval
print("=" * 60)
print("Query: What does Lilian Weng say about types of reward hacking?")
print("=" * 60)

for chunk in graph.stream(
    {
        "messages": [{
            "role": "user",
            "content": "What does Lilian Weng say about types of reward hacking?",
        }]
    }
):
    for node, update in chunk.items():
        print(f"\n--- Update from node: [{node}] ---")
        update["messages"][-1].pretty_print()

In [ ]:
# Question that can be answered directly without retrieval
print("=" * 60)
print("Query: Hello! What can you help me with?")
print("=" * 60)

for chunk in graph.stream(
    {
        "messages": [{
            "role": "user",
            "content": "Hello! What can you help me with?",
        }]
    }
):
    for node, update in chunk.items():
        print(f"\n--- Update from node: [{node}] ---")
        update["messages"][-1].pretty_print()

In [ ]:
# Another question from one of the other blog posts
print("=" * 60)
print("Query: What does Lilian Weng say about hallucinations in LLMs?")
print("=" * 60)

for chunk in graph.stream(
    {
        "messages": [{
            "role": "user",
            "content": "What does Lilian Weng say about hallucinations in LLMs?",
        }]
    }
):
    for node, update in chunk.items():
        print(f"\n--- Update from node: [{node}] ---")
        update["messages"][-1].pretty_print()